# Stonks: Algorithmic Trading with Forester + Golden Ratio

Based on paper: *Algorithmic Trading Model for Stock Price Forecasting Integrating Forester with Golden Ratio Strategy* (2024 IEEE R10-HTC, DOI: 10.1109/R10-HTC59322.2024.10778666)

In [ ]:
import sys, os
# Automatically locate the Stonks root directory
current_dir = os.getcwd()
while not os.path.exists(os.path.join(current_dir, 'model', 'src')) and current_dir != os.path.dirname(current_dir):
    current_dir = os.path.dirname(current_dir)
if current_dir not in sys.path:
    sys.path.append(current_dir)

from IPython.display import display, HTML
from model.src.data_pipeline import run_data_pipeline
from model.src.forester_model import run_forester
from model.src.golden_ratio import run_golden_ratio
from model.src.backtester import run_backtest
from model.config.settings import TICKERS, START_DATE, END_DATE
import matplotlib.pyplot as plt
import pandas as pd

### Step 1: Data Pipeline
Fetching and preprocessing raw data for RELIANCE.NS, generating technical indicators.

In [ ]:
ticker = 'RELIANCE.NS'
raw_df, X, y = run_data_pipeline(ticker, START_DATE, END_DATE)
display(X.head())

### Step 2: Forester Model Training
Training the ExtraTreesRegressor (Forester) model and evaluating performance metrics.

In [ ]:
model, model_metrics, predictions_df = run_forester(ticker, X, y)
metrics_df = pd.DataFrame([model_metrics])
display(metrics_df)

### Step 3: Actual vs Predicted Prices
Plotting the model's predictions over the dataset.

In [ ]:
plt.figure(figsize=(14, 7))
plt.plot(predictions_df.index, predictions_df['Actual'], label='Actual Price (₹)', alpha=0.6)
plt.plot(predictions_df.index, predictions_df['Predicted'], label='Predicted Price (₹)', alpha=0.8)
plt.title(f'{ticker} - Actual vs Predicted')
plt.legend()
plt.show()

### Step 4: Golden Ratio Strategy
Computing Fibonacci levels and generating trading signals.

In [ ]:
signal_df = run_golden_ratio(ticker, predictions_df)
fib_cols = [c for c in signal_df.columns if c.startswith('Fib_')]
fib_levels = {c: signal_df[c].iloc[0] for c in fib_cols}
print('Fibonacci Levels Generated:')
for k, v in fib_levels.items(): print(f'{k}: {v:.2f}')
display(signal_df[['Actual', 'Predicted', 'Signal']].head(10))

### Step 5: Signals Annotated inline
Visualizing the price chart with generated Buy/Sell signals.

In [ ]:
plt.figure(figsize=(14, 7))
plt.plot(signal_df.index, signal_df['Actual'], label='Actual Price (₹)', color='black', alpha=0.5)
buys = signal_df[signal_df['Signal'] == 'Buy']
sells = signal_df[signal_df['Signal'] == 'Sell']
plt.scatter(buys.index, buys['Actual'], marker='^', color='green', label='Buy Signal', s=100)
plt.scatter(sells.index, sells['Actual'], marker='v', color='red', label='Sell Signal', s=100)
plt.legend()
plt.title('Trading Signals')
plt.show()

### Step 6: Backtesting
Simulating trades using the signals to compute strategy metrics.

In [ ]:
portfolio_df, backtest_metrics = run_backtest(ticker, signal_df)
display(pd.DataFrame([backtest_metrics]))

### Step 7: Portfolio vs Buy-and-Hold
Comparing the algorithmic strategy against a passive hold approach.

In [ ]:
plt.figure(figsize=(14, 7))
initial_capital = portfolio_df['Capital'].iloc[0] if portfolio_df['Shares'].iloc[0] == 0 else portfolio_df['Portfolio_Value'].iloc[0]
initial_price = portfolio_df['Price'].iloc[0]
buy_hold_value = portfolio_df['Price'] * (initial_capital / initial_price)

plt.plot(portfolio_df.index, portfolio_df['Portfolio_Value'], label='Strategy Portfolio (₹)', color='purple')
plt.plot(portfolio_df.index, buy_hold_value, label='Buy & Hold (₹)', color='grey', linestyle='dashed')
plt.legend()
plt.title('Portfolio Performance')
plt.show()

### Conclusion
The Forester + Golden Ratio integration successfully generates systematic trading signals. The backtest illustrates performance relative to a buy-and-hold strategy, highlighting metrics such as Max Drawdown and Sharpe Ratio.

### Step 8: Trade Log Extraction
Extracting a detailed log of completed buy-sell pair trades.

In [ ]:
from model.src.backtester import extract_trade_log

trade_log = extract_trade_log(portfolio_df)
trade_log_df = pd.DataFrame(trade_log)
display(trade_log_df.head())

### Step 9: Live Trading Prediction
Running the live pipeline to fetch out-of-sample present data, scale safely without data leaks, and compute tomorrow's signal.

In [ ]:
from model.live import run_live

live_result = run_live(ticker)
print("Live Trade Recommendation for Today:")
for k, v in live_result.items():
    if k == 'fib_levels': continue
    print(f"{k}: {v}")